In [ ]:
'''
from google.colab import drive
drive.mount('/content/drive')
'''

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#!pip install mediapipe==0.10.21

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.3 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of jax to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of opencv-contrib-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 84.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 92.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.2/81.2 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 MB 11.0 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled p

In [ ]:
#!pip install mediapipe opencv-python numpy


In [ ]:
import cv2
import mediapipe as mp
import numpy as np
from collections import Counter, deque


In [ ]:
mp_face = mp.solutions.face_detection
face_detector = mp_face.FaceDetection(
    model_selection=0,
    min_detection_confidence=0.4
)

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)


In [ ]:
def get_index_fingertip(hand_landmarks, img_w, img_h):
    tip = hand_landmarks.landmark[8]  # index fingertip
    fx = int(tip.x * img_w)
    fy = int(tip.y * img_h)
    return fx, fy


In [ ]:
def get_face_bbox(face_detection, img_w, img_h):
    bbox = face_detection.location_data.relative_bounding_box

    x1 = int(max(0, bbox.xmin * img_w))
    y1 = int(max(0, bbox.ymin * img_h))
    x2 = int(min(img_w, (bbox.xmin + bbox.width) * img_w))
    y2 = int(min(img_h, (bbox.ymin + bbox.height) * img_h))

    return x1, y1, x2, y2


In [ ]:
def is_near_face(face_bbox, fx, fy):
    x1, y1, x2, y2 = face_bbox
    face_h = y2 - y1
    face_w = x2 - x1

    rel_y = (fy - y1) / face_h

    # Nose / mouth projection zone
    if 0.35 <= rel_y <= 0.65:
        return True

    pad_x = 0.35 * face_w
    pad_y = 0.35 * face_h

    return (
        x1 - pad_x <= fx <= x2 + pad_x and
        y1 - pad_y <= fy <= y2 + pad_y
    )


In [ ]:
def mouth_center(face_bbox):
    x1, y1, x2, y2 = face_bbox
    face_h = y2 - y1
    face_w = x2 - x1

    mx = x1 + 0.5 * face_w
    my = y1 + 0.55 * face_h   # mouth center (empirical)

    return mx, my


In [ ]:
from collections import deque
fy_history = deque(maxlen=7)


In [ ]:
def classify_face_region(face_bbox, contact_point):
    x1, y1, x2, y2 = face_bbox
    fx, fy = contact_point

    face_h = y2 - y1
    face_w = x2 - x1

    rel_y = (fy - y1) / face_h
    rel_x = (fx - x1) / face_w

    # ---------------- HEAD ----------------
    if rel_y < 0.05:
        return "hamheadtop"

    if 0.05 <= rel_y < 0.20:
        return "hamforehead"

    # ---------------- EYES / EARS ----------------
    if 0.20 <= rel_y < 0.35:
        if rel_x < 0.15 or rel_x > 0.85:
            return "hamear"
        return "hameyes"

    # ---------------- NOSE ----------------
    if 0.35 <= rel_y < 0.48:
        return "hamnose" if rel_y < 0.42 else "hamnostrils"

    # ---------------- MOUTH (FIXED) ----------------

    if 0.48 <= rel_y < 0.75:
      if 0.30 <= rel_x <= 0.70:

        fy_history.append(fy)

        if len(fy_history) >= 5:
            y_range = max(fy_history) - min(fy_history)
        else:
            y_range = 0

        # ---- TEETH: stable contact ----
        if y_range < 0.02 * face_h:
            return "hamteeth"

        # ---- LIPS: above mouth ----
        mx, my = mouth_center(face_bbox)
        if fy < my - 0.04 * face_h:
            return "hamlips"

        # ---- TONGUE: large motion ----
        return "hamtongue"

    return "hamcheek"



    # ---------------- CHIN / NECK ----------------
    if 0.72 <= rel_y < 0.82:
        return "hamchin"

    if 0.82 <= rel_y < 0.90:
        return "hamunderchin"

    return "hamneck"


In [ ]:
def stabilize_y(prev_y, curr_y, alpha=0.6):
    if prev_y is None:
        return curr_y
    return alpha * curr_y + (1 - alpha) * prev_y


In [ ]:
'''
#cap = cv2.VideoCapture("/content/drive/MyDrive/teeth_testing.mp4")

labels = []
prev_fy = None

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    img_h, img_w = frame.shape[:2]
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    face_res = face_detector.process(rgb)
    hand_res = hands.process(rgb)

    if not face_res.detections or not hand_res.multi_hand_landmarks:
        continue

    face_bbox = get_face_bbox(face_res.detections[0], img_w, img_h)
    fx, fy = get_index_fingertip(
        hand_res.multi_hand_landmarks[0], img_w, img_h
    )

    if not is_near_face(face_bbox, fx, fy):
        continue

    fy_stable = stabilize_y(prev_fy, fy)
    prev_fy = fy_stable

    # ✅ ADD THIS LINE HERE (IMPORTANT)
    fy_history.append(fy_stable)

    label = classify_face_region(face_bbox, (fx, fy_stable))
    labels.append(label)

cap.release()
'''

In [ ]:
#print(cap.isOpened())


False


In [ ]:
from collections import Counter

#print("Total frames classified:", len(labels))
#print(Counter(labels))


Total frames classified: 86
Counter({'hamteeth': 76, 'hamtongue': 5, 'hamcheek': 5})


In [ ]:

import cv2
import numpy as np
from collections import Counter

def visualize_hamnosys_video(video_path, output_path):
    cap = cv2.VideoCapture(video_path)
    out = None

    labels = []
    prev_fy = None

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        img_h, img_w = frame.shape[:2]
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        face_mesh_res = face_mesh.process(rgb)
        hand_res = hands.process(rgb)

        if not face_mesh_res.multi_face_landmarks or not hand_res.multi_hand_landmarks:
            continue

        face_bbox = get_face_bbox_from_mesh(
            face_mesh_res.multi_face_landmarks[0], img_w, img_h
        )

        fx, fy = get_index_fingertip(
            hand_res.multi_hand_landmarks[0], img_w, img_h
        )

        if not is_near_face(face_bbox, fx, fy):
            continue

        fy_stable = stabilize_y(prev_fy, fy)
        prev_fy = fy_stable
        fy_history.append(fy_stable)

        label = classify_face_region(face_bbox, (fx, fy_stable))
        labels.append(label)

        # ---------- DRAW ----------
        x1, y1, x2, y2 = face_bbox
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)
        cv2.circle(frame, (int(fx), int(fy_stable)), 6, (0,0,255), -1)

        cv2.putText(frame, f"Frame label: {label}",
                    (20, 30), cv2.FONT_HERSHEY_SIMPLEX,
                    0.8, (255,255,0), 2)

        if out is None:
            out = cv2.VideoWriter(
                output_path,
                cv2.VideoWriter_fourcc(*"mp4v"),
                cap.get(cv2.CAP_PROP_FPS),
                (img_w, img_h)
            )

        out.write(frame)

    cap.release()
    if out:
        out.release()

    counts = Counter(labels)
    final_label, freq = counts.most_common(1)[0]
    confidence = freq / len(labels)

    return final_label, round(confidence, 2)


In [ ]:
import mediapipe as mp

mp_face_mesh = mp.solutions.face_mesh

face_mesh = mp_face_mesh.FaceMesh(
    static_image_mode=False,
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)



In [ ]:
def get_face_bbox_from_mesh(face_landmarks, img_w, img_h):
    xs = [lm.x * img_w for lm in face_landmarks.landmark]
    ys = [lm.y * img_h for lm in face_landmarks.landmark]

    x1 = int(min(xs))
    y1 = int(min(ys))
    x2 = int(max(xs))
    y2 = int(max(ys))

    return x1, y1, x2, y2


In [ ]:
final_label, conf = visualize_hamnosys_video(
    "/content/drive/MyDrive/Colab Notebooks/nose_testing.mp4",
    "/content/drive/MyDrive/nose_pre.mp4"
)

#print(final_label, conf)


hamnostrils 0.6


In [ ]:
def detect_head_face_location(frame):

    img_h, img_w = frame.shape[:2]
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    face_res = face_detector.process(rgb)
    hand_res = hands.process(rgb)

    if not face_res.detections or not hand_res.multi_hand_landmarks:
        return None

    face_bbox = get_face_bbox(face_res.detections[0], img_w, img_h)

    fx, fy = get_index_fingertip(
        hand_res.multi_hand_landmarks[0], img_w, img_h
    )

    if not is_near_face(face_bbox, fx, fy):
        return None

    fy_stable = stabilize_y(None, fy)  # smoothing optional

    fy_history.append(fy_stable)

    return classify_face_region(face_bbox, (fx, fy_stable))

In [ ]:
def run_head_face_location_module(video_path):

    predictions = []

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        return {"per_frame": [], "final": None}

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        loc = detect_head_face_location(frame)   # ✅ FIXED

        if loc:
            predictions.append(loc)

    cap.release()

    if not predictions:
        return {"per_frame": [], "final": None}

    final_location = Counter(predictions).most_common(1)[0][0]

    return {
        "per_frame": predictions,
        "final": final_location
    }